# Libs

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
import joblib
from sklearn.model_selection import ParameterGrid
from tqdm.auto import tqdm

import os
import logging
import sys
sys.path.append('../')
import src.forecasting.simulations      as sim
import src.fda.kde.estimators           as kde
import src.fda.transformations.lqdt     as lqdt

import src.fda.utils                    as fdaUtils
import src.forecasting.pipelines        as fp
import src.forecasting.accuracy         as acc
import src.forecasting.cross_validation as crossVal
import src.fda.dfpc as dfpc

In [2]:
EXECUTION_DATE : str = datetime.now().strftime('%Y%m%d')
LOG_PATH       : str = f'../logs/simulations/{EXECUTION_DATE}.log'
FILES_PATH     : str = f'../data/interim/simulation/{EXECUTION_DATE}/'
DATABASES_PATH : str = f'../data/interim/simulation/{EXECUTION_DATE}/databases/'
SIM_LQD_PATH       : str = f'../data/interim/simulation/{EXECUTION_DATE}/sim_lqds/'
KDE_LQD_PATH   : str = f'../data/interim/simulation/{EXECUTION_DATE}/kde_lqds/'
ALL_KDES_PATH  : str = f'../data/interim/simulation/{EXECUTION_DATE}/all_kdes.jbl'
CV_FILES_PATH  : str = f'../data/interim/simulation/{EXECUTION_DATE}/progress/'
CV_FC_PATH     : str = f'../data/interim/simulation/{EXECUTION_DATE}/cv/'


Path(FILES_PATH).mkdir(parents=True, exist_ok=True)
Path(DATABASES_PATH).mkdir(parents=True, exist_ok=True)
Path(CV_FILES_PATH).mkdir(parents=True, exist_ok=True)
Path(SIM_LQD_PATH).mkdir(parents=True, exist_ok=True)
Path(KDE_LQD_PATH).mkdir(parents=True, exist_ok=True)
Path(CV_FC_PATH).mkdir(parents=True, exist_ok=True)

Let $\lbrace u\mapsto g(u\mid \theta; \pi) : \theta\in\Theta, \pi\in\Pi\rbrace$ be a parametric family of (deterministic) densities with parameter space $\Theta\times\Pi$, where $\Theta\subseteq\mathbb{R}^k$. 

For fixed (deterministic) model parameters $\pi\in\Pi$, $\theta^*\in \Theta$, and $k\times k$ matrices $\alpha$ and $\beta$, plus a user supplied parameter $\gamma\in\lbrace0, 1/2, 1\rbrace$, consider the model given by the following dynamic equations

\begin{equation}
\mathcal{Z}_{t-1} = \sigma(Z_s : s\leq t-1)
\end{equation}

\begin{equation}
\mathbb{P}(Z_{t}\leq z\mid \mathcal{Z}_{t-1}) = \int_{-\infty}^{z} f_t(z)\,\mathrm{d}z,\qquad f_t(z):=g(z\mid \vartheta_t;\pi)
\end{equation}

\begin{equation}
\vartheta_{t+1} - \theta^* = \alpha\, (\mathbb{E}[\nabla_t\nabla_t^{\intercal}\mid \mathcal{Z}_{t-1}]^{-\gamma}\,\nabla_t) + \beta\, (\vartheta_{t} - \theta^*)
\end{equation}

where

\begin{equation}
\nabla_t = \left.\frac{\partial \log g(u\mid\theta;\pi)}{\partial \theta}\right|_{u=Z_t,\ \theta=\vartheta_t}
\end{equation}

\begin{equation}
\vartheta_t = (m_t, \ell_t, \eta_t) = (mean_t, log(std_t), log(asymmetry_t))
\end{equation}

# Simulations

In [3]:
# Define scenarios
gas_params = {
    # Location-driven dynamics
    # "scenario_1": {
    #     "alpha": np.diag([0.08, 0.01, 0.005]),
    #     "beta": np.diag([0.90, 0.95, 0.95]),
    # },
    # Scale-driven dynamics (volatility clustering)
    "scenario_2": {
        "alpha": np.diag([0.01, 0.08, 0.01]),
        "beta": np.diag([0.95, 0.90, 0.95]),
    },
    # Shape / skewness-driven dynamics
    # "scenario_3": {
    #      "alpha": np.diag([0.01, 0.01, 0.08]),
    #       "beta": np.diag([0.95, 0.95, 0.90])
    # }
}

distribution_params = {
    "nu": [3, 8]
}

In [4]:
from itertools import product

keys = ["scenario", "nu"]
values = [list(gas_params.keys()), distribution_params["nu"]]

param_grid = []

for scenario, nu in product(*values):
    gas_cfg = gas_params[scenario]

    param_grid.append({
        "scenario": "___nu=".join([scenario, str(nu)]),
        "nu": nu,
        "alpha": gas_cfg["alpha"],
        "beta": gas_cfg["beta"]    
})

In [5]:
x = np.linspace(-10, 10, 5001)
N_REPS = 500
T = 100
N_SAMPLES = 288
total = len(param_grid) * N_REPS

sim_database = {}
progress = 0

for params in param_grid:
    scenario = params["scenario"]
    # Initialize scenario level
    sim_database[scenario] = {
        "params": params,
        "replications": {}
    }
    
    for n_rep in range(N_REPS):
        # 1. Setup Model and Simulate
        gm = sim.GasModel(alpha=params["alpha"], beta=params["beta"], nu=params["nu"])
        sim_results = gm.simulate(T=T)
        
        # 2. Get Theoretical Densities
        sim_density = gm.conditional_densities(grid=x, theta_path=sim_results["theta"])
        dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=T, freq="D")
        sim_density.columns = dates
        
        # 3. Generate Samples Efficiently
        # Collect arrays first, then create DataFrame once
        samples_list = []
        for i in range(len(sim_results["theta"])):
            # Draw n samples for the theta at time i
            sample = gm.rvs(n=N_SAMPLES, theta=sim_results["theta"][i])
            samples_list.append(sample)
        
        # Create DataFrame: each column is a time step, each row a sample
        df_samples = pd.DataFrame(np.array(samples_list).T, columns=dates)
        
        # 4. Store in Database
        sim_database[scenario]["replications"][n_rep] = {
            "theta": sim_results["theta"],
            "densities": sim_density,
            "samples": df_samples
        }

        progress += 1
        if progress % 10 == 0: # Print less frequently to keep console clean
            print(f"Progress: {progress}/{total}")

Progress: 10/1000
Progress: 20/1000
Progress: 30/1000
Progress: 40/1000
Progress: 50/1000
Progress: 60/1000
Progress: 70/1000
Progress: 80/1000
Progress: 90/1000
Progress: 100/1000
Progress: 110/1000
Progress: 120/1000
Progress: 130/1000
Progress: 140/1000
Progress: 150/1000
Progress: 160/1000
Progress: 170/1000
Progress: 180/1000
Progress: 190/1000
Progress: 200/1000
Progress: 210/1000
Progress: 220/1000
Progress: 230/1000
Progress: 240/1000
Progress: 250/1000
Progress: 260/1000
Progress: 270/1000
Progress: 280/1000
Progress: 290/1000
Progress: 300/1000
Progress: 310/1000
Progress: 320/1000
Progress: 330/1000
Progress: 340/1000
Progress: 350/1000
Progress: 360/1000
Progress: 370/1000
Progress: 380/1000
Progress: 390/1000
Progress: 400/1000
Progress: 410/1000
Progress: 420/1000
Progress: 430/1000
Progress: 440/1000
Progress: 450/1000
Progress: 460/1000
Progress: 470/1000
Progress: 480/1000
Progress: 490/1000
Progress: 500/1000
Progress: 510/1000
Progress: 520/1000
Progress: 530/1000
Pr

In [29]:
# joblib.dump(sim_database, f"{FILES_PATH}simulation_database.jbl")

simulations_database = joblib.load(f"{FILES_PATH}simulation_database.jbl")

In [177]:
# model = 'scenario_1___nu=3'
# print(sim_database[model]["params"])
# sim_database[model][10]["densities"].iloc[:,:2].plot(figsize=(15,5))
# sim_database[model][10]["samples"].iloc[:,:2].plot(kind="hist", density=True, bins=30, figsize=(15,5), alpha=.5)

# KDEs

In [16]:
# # KDE params
# # t_dfs = range(3,6)
# t_dfs = [3]

# # rot_grid = {
# #     "method": ["rot"],
# #     "kernel": ["gaussian"],
# #     # "sigma_robust": [True, False]
# # }

# adaptive_grid = {
#     "method": ["adaptive"], 
#     "kernel": ["gaussian"]
# }

# adaptive_grid_t = {
#     "method": ["adaptive"], 
#     "kernel": ["t-student"],
#     "df": [df for df in t_dfs],
# }


# density_param_grid = {}
# for grid in [adaptive_grid, adaptive_grid_t]:
#     for params in ParameterGrid(grid):
        
#         key_parts = [params["kernel"]]
#         if "method" in params: key_parts.append(params["method"])
#         if "df" in params: key_parts.append(f"df={params['df']}")
#         if params.get("sigma_robust"): key_parts.append("robust")
#         if params.get("cv") == "LOO": key_parts.append("loo")
        
#         full_model_name = "_".join(key_parts).replace(".", "")
        
#         kernel_label = params["kernel"]
#         if "df" in params:
#             kernel_label += f"+df={params['df']}"
            
#         bw_label = params.get("method", "fixed")
#         if params.get("sigma_robust"): bw_label += "_robust"
#         if params.get("cv") == "LOO": bw_label += "_loo"


#         density_param_grid[full_model_name] = {
#             "kernel": kernel_label,
#             "bandwidth": bw_label,
#             "params": params  
#         }

# print("KDE models:")
# for name, values in density_param_grid.items():
#     print("\t",name, ":", values["params"])

# print(f"Total KDE models: {len(density_param_grid.items())}")

In [7]:
# KDE params
# t_dfs = range(3,6)
t_dfs = [3]

rot_grid = {
    "method": ["rot"],
    "kernel": ["gaussian"],
    "sigma_robust": [False]
}

rot_grid_t = {
    "method": ["rot"],
    "kernel": ["t-student"],
    "df": [df for df in t_dfs],
    "sigma_robust": [False]
}

# adaptive_grid = {
#     "method": ["adaptive"], 
#     "kernel": ["gaussian"]
# }

# adaptive_grid_t = {
#     "method": ["adaptive"], 
#     "kernel": ["t-student"],
#     "df": [df for df in t_dfs],
# }


density_param_grid = {}
for grid in [rot_grid, rot_grid_t]:
    for params in ParameterGrid(grid):
        
        key_parts = [params["kernel"]]
        if "method" in params: key_parts.append(params["method"])
        if "df" in params: key_parts.append(f"df={params['df']}")
        if params.get("sigma_robust"): key_parts.append("robust")
        if params.get("cv") == "LOO": key_parts.append("loo")
        
        full_model_name = "_".join(key_parts).replace(".", "")
        
        kernel_label = params["kernel"]
        if "df" in params:
            kernel_label += f"+df={params['df']}"
            
        bw_label = params.get("method", "fixed")
        if params.get("sigma_robust"): bw_label += "_robust"
        if params.get("cv") == "LOO": bw_label += "_loo"


        density_param_grid[full_model_name] = {
            "kernel": kernel_label,
            "bandwidth": bw_label,
            "params": params  
        }

print("KDE models:")
for name, values in density_param_grid.items():
    print("\t",name, ":", values["params"])

print(f"Total KDE models: {len(density_param_grid.items())}")

KDE models:
	 gaussian_rot : {'kernel': 'gaussian', 'method': 'rot', 'sigma_robust': False}
	 t-student_rot_df=3 : {'df': 3, 'kernel': 't-student', 'method': 'rot', 'sigma_robust': False}
Total KDE models: 2


In [9]:
total = len(sim_database) * N_REPS * len(density_param_grid)
pbar = tqdm(total=total, desc="Total CV Progress")

kde_databases = {}
# progress = 0
for scenario, database in sim_database.items():
    print(f"Processing configuration for {database['params']}")
    kde_databases[scenario] = {}
    sim_reps_database = sim_database[scenario]["replications"]
    
    for n_rep, df in sim_reps_database.items():
        print(f"\t{n_rep}")
        returns_df = sim_reps_database[n_rep]['samples']
        kde_databases[scenario][n_rep] = {}
        
        for kde_bw_name, kde_bw_params in density_param_grid.items():
            print(f"\t simulation n. = {n_rep}: {kde_bw_name}")
            
            # bandwidths
            df_h = kde.df_bandwidth_selector(returns_df, **kde_bw_params["params"])    
            kde_params = {k: v for k, v in kde_bw_params["params"].items() if k in ['kernel', 'df']}
            
            # kdes
            df_grids, df_densities = kde.df_to_kde(
                X=returns_df, 
                h=df_h, 
                normalize_densities=False,
                **kde_params
            )

            rep_result = {
                        "scenario":     scenario,
                        "n_rep":        n_rep,
                        "model_name":   kde_bw_name,
                        "kde_params":   kde_bw_params["kernel"],
                        "kernel":       kde_bw_params["params"]["kernel"],
                        "bw_params":    kde_bw_params["bandwidth"],
                        "bw_method":    kde_bw_params["params"]["method"],
                        "df_h":         df_h,
                        "df_support":   df_grids,
                        "df_densities": df_densities
                }

            # kde_databases[scenario][n_rep][kde_bw_name] = rep_result
            
            joblib.dump(rep_result, f"{CV_FILES_PATH}{scenario}___rep_{n_rep}___kde_{kde_bw_name}.jbl")

            del df_h
            del df_grids
            del df_densities
            del kde_params

            # progress += 1
            # print(f"{progress}/{total}")
            pbar.update(1)


Total CV Progress:   0%|          | 0/2000 [00:00<?, ?it/s]

Processing configuration for {'scenario': 'scenario_2___nu=3', 'nu': 3, 'alpha': array([[0.01, 0.  , 0.  ],
       [0.  , 0.08, 0.  ],
       [0.  , 0.  , 0.01]]), 'beta': array([[0.95, 0.  , 0.  ],
       [0.  , 0.9 , 0.  ],
       [0.  , 0.  , 0.95]])}
	0
	 simulation n. = 0: gaussian_rot
	 simulation n. = 0: t-student_rot_df=3
	1
	 simulation n. = 1: gaussian_rot
	 simulation n. = 1: t-student_rot_df=3
	2
	 simulation n. = 2: gaussian_rot
	 simulation n. = 2: t-student_rot_df=3
	3
	 simulation n. = 3: gaussian_rot
	 simulation n. = 3: t-student_rot_df=3
	4
	 simulation n. = 4: gaussian_rot
	 simulation n. = 4: t-student_rot_df=3
	5
	 simulation n. = 5: gaussian_rot
	 simulation n. = 5: t-student_rot_df=3
	6
	 simulation n. = 6: gaussian_rot
	 simulation n. = 6: t-student_rot_df=3
	7
	 simulation n. = 7: gaussian_rot
	 simulation n. = 7: t-student_rot_df=3
	8
	 simulation n. = 8: gaussian_rot
	 simulation n. = 8: t-student_rot_df=3
	9
	 simulation n. = 9: gaussian_rot
	 simulation n.

In [30]:
# kdes_paths = [''.join([CV_FILES_PATH, x]) for x in os.listdir(CV_FILES_PATH) if "scenario" in x]
kde_jbls_path = CV_FILES_PATH
kdes_paths = [''.join([kde_jbls_path, x]) for x in os.listdir(kde_jbls_path)]

kde_addresses = []
for path_name in kdes_paths:
    obj = joblib.load(path_name)
    kde_address = {
        "scenario":   obj["scenario"],
        "n_rep":      obj["n_rep"],
        "model_name": obj["model_name"],
        "address":    path_name
    }
    kde_addresses.append(kde_address)

kde_addresses = sorted(
    kde_addresses,
    key=lambda x: (x["scenario"], x["n_rep"], x["model_name"])
)

kde_index = {
    (d["scenario"], d["n_rep"], d["model_name"]): d["address"]
    for d in kde_addresses
}

# $Y_t=\Lambda(\hat{f_t})$

In [19]:
n_kde_addresses = len(kde_addresses)
pbar = tqdm(total=n_kde_addresses, desc="Total LQD(fhat) Progress")

for kde in kde_addresses:
    scenario_kde_dict = joblib.load(kde["address"])
    id_scenario_kde     = scenario_kde_dict["scenario"]
    n_rep               = scenario_kde_dict["n_rep"]
    model_name_path     = scenario_kde_dict['model_name']
    model_name          = scenario_kde_dict['model_name'].replace("_", " ")
    # carregar KDE
    # transformar para L2
    # salvar

    mlqdt = lqdt.mLQDT()
    model_lqd = mlqdt.transform(
        densities=scenario_kde_dict["df_densities"],
        densities_supports=scenario_kde_dict["df_support"], 
        verbose=False
        )
    # self.model_lqd.densities_to_lqdensities(verbose=False)
    
    # 2. L2 Expansion (K_dFPC)
    Y_t  = model_lqd.lqd.copy()
    Y_t.index = model_lqd.lqd_support
    
    # KdFPC_kwargs.update({
    #     "u": lqd_support,
    #     "du": model_lqd.du
    # })
    
    # model_kdfpc = dfpc.K_dFPC(lqd_values)
    # model_kdfpc.fit(**KdFPC_kwargs)

    # Y_t = pd.DataFrame(model_kdfpc.Y)
    # Y_t.columns = df_densities.columns
    # Y_t.index   = model_lqd.lqd_support

    kde_mlqdt_result = {
                "scenario":     kde["scenario"],
                "n_rep":        kde["n_rep"],
                "model_name":   kde["model_name"],
                "kde_mlqdt":    Y_t
        }
    
    joblib.dump(kde_mlqdt_result, f"{KDE_LQD_PATH}lqd___scenario_{id_scenario_kde}___rep_{n_rep}___kde_{model_name_path}.jbl")

    pbar.update(1)

Total LQD(fhat) Progress:   0%|          | 0/2000 [00:00<?, ?it/s]

/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: overflow encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: divide by zero encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]


In [31]:
# kdes_lqd_paths = [''.join([KDE_LQD_PATH, x]) for x in os.listdir(KDE_LQD_PATH) if "scenario" in x]
kdes_lqd_paths = [''.join([KDE_LQD_PATH, x]) for x in os.listdir(KDE_LQD_PATH)]


kdes_lqd_addresses = []
for path_name in kdes_lqd_paths:
    obj = joblib.load(path_name)
    kde_address = {
        "scenario":   obj["scenario"],
        "n_rep":      obj["n_rep"],
        "model_name": obj["model_name"],
        "address":    path_name
    }
    kdes_lqd_addresses.append(kde_address)

kde_lqd_index = {
    (d["scenario"], d["n_rep"], d["model_name"]): d["address"]
    for d in kdes_lqd_addresses
}

# Cross-validation

In [32]:
# 1. Setup Logger
logger = logging.getLogger("Density_Estimation")
logger.setLevel(logging.DEBUG)

# 2. Setup File Handler
file_handler = logging.FileHandler(LOG_PATH)
file_handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
logger.addHandler(file_handler)

# 3. CAPTURE WARNINGS: This handles "Forecast with mean" style warnings
logging.captureWarnings(True)

In [33]:
KdFPC_kwargs = {
    "p": 5,
    "select_ncomp": "variance",
    # "dimension": 3
}

In [34]:
initial_window = 99
horizon = 1

total_models = len(kde_addresses)
pbar = tqdm(total=total_models, desc="Total CV Progress")

logger.info("\nInitiating cross validation...")
for kde_address in kde_addresses:
    print(kde_address)
    scenario_kde_dict = joblib.load(kde_address["address"])

    # KDE database
    id_scenario_kde     = scenario_kde_dict["scenario"]
    sim_info            = simulations_database[id_scenario_kde]["params"]
    n_rep               = scenario_kde_dict["n_rep"]
    model_name_path     = scenario_kde_dict['model_name']
    model_name          = scenario_kde_dict['model_name'].replace("_", " ")
    logger.info(
            f"Scenario: {id_scenario_kde} | n_rep: {n_rep} | KDE model: {model_name}"
            )

    df_support, df_densities = scenario_kde_dict["df_support"], scenario_kde_dict["df_densities"]

    # Simulations database
    sim_densities         = simulations_database[id_scenario_kde]["replications"][n_rep]["densities"]
    sim_densities_supp    = sim_densities.copy()
    sim_densities_supp.loc[:,:] = sim_densities_supp.index.to_numpy()[:, None]


    # L2 database
    kde_lqd_address       = kde_lqd_index.get((id_scenario_kde, n_rep, scenario_kde_dict['model_name']))
    scenario_kde_lqd      = joblib.load(kde_lqd_address)["kde_mlqdt"]
    scenario_kde_lqd_supp = scenario_kde_lqd.copy()
    scenario_kde_lqd_supp.loc[:,:] = scenario_kde_lqd_supp.index.to_numpy()[:, None]
    
    # Cross-validation
    windows = crossVal.expanding_window_cv(df_densities.shape[1], h=horizon, initial_window=initial_window)
#     curves_hist = []
    measures = []
    for fold, window in enumerate(windows):
            fold += 1
            print(f"\t\t>>> cv {fold}/{len(windows)}")
            idx_train      = window[0]
            idx_test       = window[1]
            test_date      = df_densities.columns[idx_test]
            
            #------------ target variables ------------#
            Y_t_support, Y_t                   = scenario_kde_lqd_supp.loc[:, test_date], scenario_kde_lqd.loc[:, test_date]
            # lambda(f_{n+1})
            # X_boot
            f_hat_t_supp, f_hat_t = df_support.loc[:, test_date], df_densities.loc[:, test_date]
            f_t_support,  f_t     = sim_densities_supp.loc[:, test_date], sim_densities.loc[:, test_date]
            # lambda^{-1}(X_boot)
            #-----------------------------------------#

            # Train-test split
            kde_train_support, kde_train = df_support.iloc[:,idx_train], df_densities.iloc[:,idx_train]
            
            # Forecasting
            forecaster = fp.DensityForecaster(kdfpc_kwargs=KdFPC_kwargs, maxlags=10)
            forecaster.fit(kde_train, kde_train_support)
            mdfpc_fc = forecaster.predict(horizon=1, var_lags=None, forecast_index=test_date)

            # Predictions
            Y_hat_t = mdfpc_fc["future_L2_curves"]

            lambda_inv_Y_hat_t_supp=  mdfpc_fc["future_supports"]
            lambda_inv_Y_hat_t =      mdfpc_fc["future_densities"]

            #------------ common grid ------------#
            # \hat{Y}_{n+1} x  Y_{n+1}
            acc_measures = acc.overall_measures(forecast=Y_hat_t, test=Y_t)
            temp_df = pd.DataFrame({
                    "support":  Y_t.index,
                    "actual":   Y_t.iloc[:, 0].values,
                    "forecast": Y_hat_t.iloc[:, 0].values,
                    "date":     test_date[0],
                    "fold":     fold
                    }).set_index(["fold", "date", "support"])
            measures.append({
                    "scenario":   id_scenario_kde,
                    "n_rep":      n_rep,
                    "model_name": model_name,
                    "fold":       fold,
                    "comparison": "yHatFc_Y",
                    "fc_date":    test_date[0],
                    **acc_measures,
                    "curves":     temp_df
            })
            # \Lambda^{-1}(\hat{Y}_{n+1}) x  \hat{f}_{n+1}
            df_supp, df_kde, df_fc = fdaUtils.align_densities(
                                            f_hat_t_supp, 
                                            f_hat_t, 
                                            lambda_inv_Y_hat_t_supp, 
                                            lambda_inv_Y_hat_t, 
                                            lambda_inv_Y_hat_t.columns
                                            )
            acc_measures = acc.overall_measures(forecast=df_fc, test=df_kde)
            temp_df = pd.DataFrame({
                    "support":  df_support.iloc[:,0].values,
                    "actual":   df_kde.iloc[:, 0].values,
                    "forecast": df_fc.iloc[:, 0].values,
                    "date":     test_date[0],
                    "fold":     fold
                    }).set_index(["fold", "date", "support"])
            measures.append({
                    "scenario":   id_scenario_kde,
                    "n_rep":      n_rep,
                    "model_name": model_name,
                    "fold":       fold,
                    "comparison": "fHatFc_fHat",
                    "fc_date":    test_date[0],
                    **acc_measures,
                    "curves":     temp_df
            })
            # \Lambda^{-1}(\hat{Y}_{n+1}) x  f_{n+1}
            df_supp, df_kde, df_fc = fdaUtils.align_densities(
                                            f_t_support, 
                                            f_t, 
                                            lambda_inv_Y_hat_t_supp, 
                                            lambda_inv_Y_hat_t, 
                                            lambda_inv_Y_hat_t.columns
                                            )
            acc_measures = acc.overall_measures(forecast=df_fc, test=df_kde)
            temp_df = pd.DataFrame({
                    "support":  df_support.iloc[:,0].values,
                    "actual":   df_kde.iloc[:, 0].values,
                    "forecast": df_fc.iloc[:, 0].values,
                    "date":     test_date[0],
                    "fold":     fold
                    }).set_index(["fold", "date", "support"])
            measures.append({
                    "scenario":   id_scenario_kde,
                    "n_rep":      n_rep,
                    "model_name": model_name,
                    "fold": fold,
                    "comparison": "fHatTp1Fc_f",
                    "fc_date":    test_date[0],
                    **acc_measures,
                    "curves":     temp_df
            })
    joblib.dump(measures, f"{CV_FC_PATH}cvFc___scenario_{id_scenario_kde}___rep_{n_rep}___kde_{model_name_path}.jbl")
    pbar.update(1)

Total CV Progress:   0%|          | 0/2000 [00:00<?, ?it/s]

{'scenario': 'scenario_2___nu=3', 'n_rep': 0, 'model_name': 'gaussian_rot', 'address': '../data/interim/simulation/20260503/progress/scenario_2___nu=3___rep_0___kde_gaussian_rot.jbl'}
		>>> cv 1/1
{'scenario': 'scenario_2___nu=3', 'n_rep': 0, 'model_name': 't-student_rot_df=3', 'address': '../data/interim/simulation/20260503/progress/scenario_2___nu=3___rep_0___kde_t-student_rot_df=3.jbl'}
		>>> cv 1/1
{'scenario': 'scenario_2___nu=3', 'n_rep': 1, 'model_name': 'gaussian_rot', 'address': '../data/interim/simulation/20260503/progress/scenario_2___nu=3___rep_1___kde_gaussian_rot.jbl'}
		>>> cv 1/1
{'scenario': 'scenario_2___nu=3', 'n_rep': 1, 'model_name': 't-student_rot_df=3', 'address': '../data/interim/simulation/20260503/progress/scenario_2___nu=3___rep_1___kde_t-student_rot_df=3.jbl'}
		>>> cv 1/1
{'scenario': 'scenario_2___nu=3', 'n_rep': 2, 'model_name': 'gaussian_rot', 'address': '../data/interim/simulation/20260503/progress/scenario_2___nu=3___rep_2___kde_gaussian_rot.jbl'}
		>>